# Module 6: Training a Diffusion Model

Welcome to the hands-on core of this course! We're going to take everything we've built so far — the U-Net from Module 4, the noise schedules from Module 5 — and wire them into a complete **DDPM training pipeline**. By the end, you'll have a trained model on MNIST and a saved checkpoint ready for sampling in Module 7.

Here's what we'll work through together:

- Loading and preprocessing image data for diffusion training
- Implementing DDPM Algorithm 1 (the training loop)
- Understanding how timestep sampling and noise prediction work
- Using **EMA** (Exponential Moving Average) for better sample quality
- Stabilizing training with learning rate schedules
- Recognizing and debugging common training pitfalls

**Estimated time:** 3–4 hours (including training time)

**Key papers:**

- **DDPM** — Ho et al. 2020. [arxiv.org/abs/2006.11239](https://arxiv.org/abs/2006.11239) — Algorithm 1
- **Improved DDPM** — Nichol & Dhariwal 2021. [arxiv.org/abs/2102.09672](https://arxiv.org/abs/2102.09672)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import math, os, copy
from typing import Dict, Tuple, Optional, List

# Shared utilities (built in earlier modules)
import sys
sys.path.insert(0, '.')
from utils.data import get_device, get_mnist_dataloader
from utils.visualization import denormalize, show_images, set_style, plot_loss_curve
from utils.unet import UNet
from utils.schedule import linear_schedule, cosine_schedule
from utils.diffusion import q_sample, prepare_schedule, train_step, ddpm_sample

torch.manual_seed(42)
set_style()
device = get_device()
print(f"Using device: {device}")

---
## UNet Architecture

Let's start by loading the UNet we built in Module 4 (now living in `utils/unet.py`). This ensures architectural consistency — the same model we designed and tested is the one we'll train here, and the checkpoint will load cleanly in Module 7 for sampling.

The UNet takes three inputs:

- **`x`** — noisy image, shape `(B, C, H, W)`
- **`t`** — integer timesteps, shape `(B,)`
- **`class_label`** (optional) — integer class labels, shape `(B,)`, for conditional generation

Key architectural choices (see Module 4 for full details):

| Component | Details |
|---|---|
| Activation | SiLU throughout |
| Time embedding | Sinusoidal positional encoding fed through a 2-layer MLP |
| Skip connections | All ResBlock outputs + downsample outputs stored and consumed by decoder |
| Attention | Custom spatial self-attention at specified resolutions |
| Upsampling | Nearest-neighbor upsample + Conv2d (not transposed conv) |

In [ ]:
# Quick shape check — does our UNet take in and output the right shapes?
model = UNet(image_channels=1, base_channels=64, channel_mults=(1, 2, 4)).to(device)
dummy_x = torch.randn(2, 1, 28, 28, device=device)
dummy_t = torch.randint(0, 1000, (2,), device=device)
out = model(dummy_x, dummy_t)
print(f"Input shape:  {dummy_x.shape}")   # (2, 1, 28, 28)
print(f"Output shape: {out.shape}")        # (2, 1, 28, 28)
total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,}")
del model, dummy_x, dummy_t, out  # free memory

---
## Noise Schedules

Next, let's load the noise schedules from `utils/schedule.py` (built in Module 5). The schedule controls **how much noise gets added at each timestep**.

| Schedule | Formula | Behavior |
|---|---|---|
| Linear | `beta_t` linearly from `beta_start` to `beta_end` | Simple, but too aggressive at high `t` |
| Cosine | `alpha_bar_t = f(t)/f(0)` where `f(t) = cos((t/T + s)/(1+s) * pi/2)^2` | Smoother, better for small images |

Both return a dictionary of pre-computed tensors that we index by timestep during training. Think of this dictionary as a **lookup table** — for any timestep `t`, we can instantly grab the right scaling factors without recomputing them.

In [ ]:
# Compare schedules visually
from utils.visualization import plot_schedule

sched_lin = linear_schedule(1000)
sched_cos = cosine_schedule(1000)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
ts = np.arange(1000)
axes[0].plot(ts, sched_lin['alphas_cumprod'].numpy(), label='Linear')
axes[0].plot(ts, sched_cos['alphas_cumprod'].numpy(), label='Cosine')
axes[0].set_xlabel('Timestep t')
axes[0].set_ylabel('alpha_bar_t')
axes[0].set_title('Cumulative Signal Retention')
axes[0].legend()

axes[1].plot(ts, sched_lin['betas'].numpy(), label='Linear')
axes[1].plot(ts, sched_cos['betas'].numpy(), label='Cosine')
axes[1].set_xlabel('Timestep t')
axes[1].set_ylabel('beta_t')
axes[1].set_title('Per-Step Noise Rate')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"Linear  — alpha_bar at t=500: {sched_lin['alphas_cumprod'][500]:.4f}")
print(f"Cosine  — alpha_bar at t=500: {sched_cos['alphas_cumprod'][500]:.4f}")

---
## 6.1 — Data Loading and Preprocessing

Diffusion models operate on images normalized to **[-1, 1]**. Why this range?

- The forward process adds Gaussian noise (mean 0), so the data should be **centered at 0**
- At `t = T`, the noisy image is approximately `N(0, I)` — this lives in the same range as the data when data is in [-1, 1]
- A symmetric range keeps the loss landscape balanced

We'll use MNIST at 28×28 for feasible training on CPU/MPS. Each image tensor has shape `(1, 28, 28)` — one channel, 28 pixels tall, 28 pixels wide. Our `get_mnist_dataloader()` from `utils/data.py` handles all the transforms for us.

In [ ]:
# Load MNIST using our shared utility
train_loader = get_mnist_dataloader(batch_size=64)

# Verify: show a batch
sample_batch, sample_labels = next(iter(train_loader))
print(f"Batch shape: {sample_batch.shape}")     # (64, 1, 28, 28)
print(f"Value range: [{sample_batch.min():.2f}, {sample_batch.max():.2f}]")

show_images(sample_batch[:8], nrow=8, title='Sample Training Images (denormalized)')

### Exercise 6.1: Verify Normalization

Let's make sure our data pipeline is working correctly before we start training.

**Your task:** Iterate over the first 10 batches and confirm:

1. The minimum value across all batches is approximately **-1.0**
2. The maximum value across all batches is approximately **1.0**
3. The mean is **negative** (MNIST is mostly black background mapped to -1, so expect around **-0.74**)

- Print the global min, max, and mean
- If any of these are wildly off, your transforms are wrong and training will fail silently

In [ ]:
# YOUR CODE HERE — Exercise 6.1

global_min = float('inf')
global_max = float('-inf')
running_sum = 0.0
running_count = 0

for i, (batch, _) in enumerate(train_loader):
    if i >= 10:
        break
    # ===================== YOUR CODE HERE =====================
    pass  # Track global_min, global_max, running_sum, running_count
    # ====================== END YOUR CODE ======================

global_mean = running_sum / max(running_count, 1)

print(f"Global min:  {global_min:.4f}  (expect ~ -1.0)")
print(f"Global max:  {global_max:.4f}  (expect ~  1.0)")
print(f"Global mean: {global_mean:.4f}  (expect ~ -0.74)")

# Tests — run this cell to check your work
if running_count == 0:
    print("\n⚠️  Looks like you haven\'t filled in the exercise yet — replace the `pass` above with your code!")
else:
    assert global_min < -0.9, f"Min is {global_min:.4f} — did you forget to iterate over batches?"
    assert global_max > 0.9, f"Max is {global_max:.4f} — are your images normalized to [-1, 1]?"
    assert -0.9 < global_mean < -0.5, f"Mean is {global_mean:.4f} — MNIST mean should be around -0.74 since most pixels are black background. Did you normalize to [-1, 1]?"
    print("Normalization verification ✓")

In [ ]:
# ✅ SOLUTION — try the exercise above before running this

global_min = float('inf')
global_max = float('-inf')
running_sum = 0.0
running_count = 0

for i, (batch, _) in enumerate(train_loader):
    if i >= 10:
        break
    global_min = min(global_min, batch.min().item())
    global_max = max(global_max, batch.max().item())
    running_sum += batch.sum().item()
    running_count += batch.numel()

global_mean = running_sum / running_count

print(f"Global min:  {global_min:.4f}  (expect ~ -1.0)")
print(f"Global max:  {global_max:.4f}  (expect ~  1.0)")
print(f"Global mean: {global_mean:.4f}  (expect ~ -0.74)")

assert global_min < -0.9, f"Min is {global_min:.4f} — did you forget to iterate over batches?"
assert global_max > 0.9, f"Max is {global_max:.4f} — are your images normalized to [-1, 1]?"
assert -0.9 < global_mean < -0.5, f"Mean is {global_mean:.4f} — MNIST mean should be around -0.74 since most pixels are black background. Did you normalize to [-1, 1]?"
print("Normalization verification ✓")

---
## 6.2 — The Training Algorithm (DDPM Algorithm 1)

Here's the remarkable thing about DDPM — the entire training loop fits in a few lines. From Ho et al. 2020, Algorithm 1:

```
repeat:
  x_0 ~ q(x_0)                                          # sample a clean image
  t   ~ Uniform({1, ..., T})                             # random timestep
  eps ~ N(0, I)                                          # sample noise
  x_t = sqrt(alpha_bar_t) * x_0 + sqrt(1 - alpha_bar_t) * eps   # forward diffusion
  loss = || eps - eps_theta(x_t, t) ||^2                 # noise prediction MSE
  gradient step on loss
until converged
```

In plain language: we take a clean image, pick a random noise level, corrupt the image with that much noise, and then ask the network to predict what noise we added.

**The loss asks one simple question: how well did the network predict the noise we added?**

| Symbol | Meaning |
|---|---|
| `x_0` | Clean training image from dataset |
| `t` | Randomly chosen noise level (higher = noisier) |
| `eps` | The ground-truth noise we added |
| `x_t` | The noisy image at timestep `t` |
| `eps_theta(x_t, t)` | Our UNet's prediction of what noise was added |
| `loss` | How wrong the prediction is — simple MSE |

We already have `q_sample()` and `train_step()` in `utils/diffusion.py` (built from the math in Module 5). Let's use them.

In [ ]:
# Quick demo: one forward diffusion step using q_sample from utils
schedule = cosine_schedule(1000)
sched_device = prepare_schedule(schedule, device)

demo_batch, _ = next(iter(train_loader))
demo_batch = demo_batch.to(device)  # (64, 1, 28, 28)
t = torch.randint(0, 1000, (demo_batch.shape[0],), device=device)  # (64,)

x_t, noise = q_sample(demo_batch, t, sched_device)  # (64, 1, 28, 28) each

print(f"Clean images x_0: {demo_batch.shape}")
print(f"Noisy images x_t: {x_t.shape}")
print(f"Noise eps:         {noise.shape}")
print(f"Timestep range:    [{t.min().item()}, {t.max().item()}]")

### Exercise 6.2: Run One Training Step

Let's make sure everything connects — model, optimizer, schedule, data.

**Your task:** Instantiate a fresh UNet and Adam optimizer (`lr=2e-4`). Run a single training step on one batch using `train_step()` from utils.

Verify that:

1. The loss is a **finite** scalar
2. The loss is approximately **1.0**

- Why ~1.0? An untrained model outputs essentially random noise predictions. Since the target noise is sampled from `N(0, I)`, the expected MSE between two independent standard-normal tensors is approximately 1.

In [ ]:
# YOUR CODE HERE — Exercise 6.2

torch.manual_seed(42)

# ===================== YOUR CODE HERE =====================
test_model = None    # Create a UNet with image_channels=1, base_channels=64, channel_mults=(1, 2, 4)
test_optimizer = None  # Create an Adam optimizer with lr=2e-4
# ====================== END YOUR CODE ======================

test_batch, _ = next(iter(train_loader))
test_batch = test_batch.to(device)
loss_val = train_step(test_model, test_batch, test_optimizer, sched_device)

print(f"Single-step loss: {loss_val:.4f}")

# Tests — run this cell to check your work
assert test_model is not None, "test_model is None — did you create the UNet?"
assert test_optimizer is not None, "test_optimizer is None — did you create the optimizer?"
assert math.isfinite(loss_val), f"Loss is {loss_val} — check that your model is on the correct device"
assert 0.3 < loss_val < 3.0, f"Loss {loss_val} is far from 1.0 — is the schedule on the right device?"
print("Single training step ✓")

del test_model, test_optimizer

In [ ]:
# ✅ SOLUTION — try the exercise above before running this

torch.manual_seed(42)
test_model = UNet(image_channels=1, base_channels=64, channel_mults=(1, 2, 4)).to(device)
test_optimizer = torch.optim.Adam(test_model.parameters(), lr=2e-4)

test_batch, _ = next(iter(train_loader))
test_batch = test_batch.to(device)
loss_val = train_step(test_model, test_batch, test_optimizer, sched_device)

print(f"Single-step loss: {loss_val:.4f}")
assert math.isfinite(loss_val), f"Loss is {loss_val} — check that your model is on the correct device"
assert 0.3 < loss_val < 3.0, f"Loss {loss_val} is far from 1.0 — is the schedule on the right device?"
print("Single training step ✓")

del test_model, test_optimizer  # free memory

---
## 6.3 — Random Timestep Sampling

**Uniform sampling** draws `t ~ Uniform({0, ..., T-1})` each step. This is what DDPM uses, and it works well in practice.

**Importance sampling** is an alternative: some timesteps contribute more to the loss than others. If we knew the per-timestep loss `L(t)`, we could sample `t` proportionally to `L(t)` and reweight the gradient to keep it unbiased. Nichol & Dhariwal (2021) explored this but found uniform sampling with enough steps works comparably.

Let's visualize how loss varies by timestep for an untrained model. You'll notice:
- **Low timesteps** (almost clean images) — easier to predict
- **High timesteps** (nearly pure noise) — harder to predict

In [ ]:
torch.manual_seed(42)
probe_model = UNet(image_channels=1, base_channels=64, channel_mults=(1, 2, 4)).to(device)
schedule = cosine_schedule(1000)
sched_device = prepare_schedule(schedule, device)

probe_batch, _ = next(iter(train_loader))  # (64, 1, 28, 28)
probe_batch = probe_batch.to(device)

# Measure loss at specific timesteps
timestep_values = list(range(0, 1000, 50))
losses_by_t = []

probe_model.eval()
with torch.no_grad():
    for t_val in timestep_values:
        t_tensor = torch.full((probe_batch.shape[0],), t_val, device=device, dtype=torch.long)
        x_t, noise = q_sample(probe_batch, t_tensor, sched_device)
        noise_pred = probe_model(x_t, t_tensor)
        loss = F.mse_loss(noise_pred, noise).item()
        losses_by_t.append(loss)

plt.figure(figsize=(8, 4))
plt.plot(timestep_values, losses_by_t, 'o-', markersize=3)
plt.xlabel('Timestep t')
plt.ylabel('MSE Loss')
plt.title('Loss vs. Timestep (untrained model, cosine schedule)')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Loss at t=0:   {losses_by_t[0]:.4f}  (almost clean — easy)")
print(f"Loss at t=500: {losses_by_t[10]:.4f}")
print(f"Loss at t=950: {losses_by_t[-1]:.4f}  (nearly pure noise — hard)")

del probe_model

### Exercise 6.3: Implement Importance-Weighted Timestep Sampling

This is a more advanced exercise. You'll implement a sampler that draws timesteps proportionally to their loss — so the model spends more training effort on the timesteps where it's struggling.

**Your task:** Implement `ImportanceTimestepSampler` with:

1. A `update(timesteps, losses)` method that maintains running average losses in `T` bins
2. A `sample(batch_size)` method that samples timesteps proportionally to `loss_per_bin + epsilon`
3. The `sample` method also returns importance weights `w_t = 1 / (T * p_t)` for unbiased gradient estimation

- Use exponential moving average for the running loss estimates
- You don't need to integrate this into training — just implement the sampling logic

In [ ]:
# YOUR CODE HERE — Exercise 6.3

class ImportanceTimestepSampler:
    """Importance-weighted timestep sampling based on running loss estimates."""

    def __init__(self, num_timesteps: int = 1000, epsilon: float = 1e-3) -> None:
        self.num_timesteps = num_timesteps
        self.epsilon = epsilon
        self.loss_bins = torch.ones(num_timesteps)  # initialize uniform
        self.counts = torch.zeros(num_timesteps)

    def update(self, timesteps: torch.Tensor, losses: torch.Tensor) -> None:
        """Update running loss estimates with observed (timestep, loss) pairs."""
        # ===================== YOUR CODE HERE =====================
        pass  # Update self.loss_bins using exponential moving average
        # ====================== END YOUR CODE ======================

    def sample(self, batch_size: int) -> Tuple[torch.Tensor, torch.Tensor]:
        """Sample timesteps proportional to loss + epsilon.
        
        Returns:
            timesteps: (B,) sampled timestep indices
            weights: (B,) importance weights for unbiased gradients
        """
        # ===================== YOUR CODE HERE =====================
        timesteps = torch.zeros(batch_size, dtype=torch.long)  # Replace
        weights = torch.ones(batch_size)  # Replace
        # ====================== END YOUR CODE ======================
        return timesteps, weights


# Tests — run this cell to check your work
sampler = ImportanceTimestepSampler(num_timesteps=1000)
fake_ts = torch.arange(0, 1000, 10)
fake_losses = torch.linspace(0.5, 2.0, len(fake_ts))
sampler.update(fake_ts, fake_losses)

sampled_ts, sampled_weights = sampler.sample(10000)
assert sampled_ts.shape == (10000,), f"Expected shape (10000,) but got {sampled_ts.shape} — check that sample() returns a 1-D tensor"
assert sampled_weights.shape == (10000,), f"Expected shape (10000,) but got {sampled_weights.shape} — weights should match timesteps shape"
assert sampled_ts.min() >= 0, f"Timestep {sampled_ts.min()} is negative — are you sampling from the right range?"
assert sampled_ts.max() < 1000, f"Timestep {sampled_ts.max()} >= 1000 — are you using num_timesteps as the upper bound?"
# High-loss timesteps should be sampled more often
high_frac = (sampled_ts > 500).float().mean().item()
assert high_frac > 0.4, f"Only {high_frac:.1%} of samples are from high-loss bins — sampling isn't weighted"
mean_weight = sampled_weights.mean().item()
assert 0.5 < mean_weight < 2.0, f"Mean weight is {mean_weight:.2f} — should be ~1.0 for unbiased estimation"
print(f"Mean weight: {mean_weight:.4f} (should be ~1.0)")
print("Importance sampling ✓")

In [ ]:
# ✅ SOLUTION — try the exercise above before running this

class ImportanceTimestepSampler:
    """Importance-weighted timestep sampling based on running loss estimates."""

    def __init__(self, num_timesteps: int = 1000, epsilon: float = 1e-3) -> None:
        self.num_timesteps = num_timesteps
        self.epsilon = epsilon
        # Running average of losses per timestep bin
        self.loss_bins = torch.ones(num_timesteps)  # initialize uniform
        self.counts = torch.zeros(num_timesteps)

    def update(self, timesteps: torch.Tensor, losses: torch.Tensor) -> None:
        """Update running loss estimates with observed (timestep, loss) pairs."""
        for t_val, loss_val in zip(timesteps.cpu(), losses.cpu()):
            t_idx = t_val.long().item()
            self.counts[t_idx] += 1
            # Exponential moving average
            alpha = 1.0 / self.counts[t_idx].item()
            self.loss_bins[t_idx] = (1 - alpha) * self.loss_bins[t_idx] + alpha * loss_val.item()

    def sample(self, batch_size: int) -> Tuple[torch.Tensor, torch.Tensor]:
        """Sample timesteps proportional to loss + epsilon.
        
        Returns:
            timesteps: (B,) sampled timestep indices
            weights: (B,) importance weights for unbiased gradients
        """
        probs = self.loss_bins + self.epsilon          # (T,)
        probs = probs / probs.sum()                    # normalize
        timesteps = torch.multinomial(probs, batch_size, replacement=True)  # (B,)
        # Importance weight: w_t = 1 / (T * p_t)
        weights = 1.0 / (self.num_timesteps * probs[timesteps])  # (B,)
        return timesteps, weights


# Demonstrate
sampler = ImportanceTimestepSampler(num_timesteps=1000)

# Simulate: pretend high timesteps have higher loss
fake_ts = torch.arange(0, 1000, 10)
fake_losses = torch.linspace(0.5, 2.0, len(fake_ts))
sampler.update(fake_ts, fake_losses)

sampled_ts, sampled_weights = sampler.sample(10000)

plt.figure(figsize=(8, 3))
plt.hist(sampled_ts.numpy(), bins=50, density=True, alpha=0.7)
plt.xlabel('Timestep')
plt.ylabel('Sampling density')
plt.title('Importance sampling: higher-loss timesteps sampled more often')
plt.tight_layout()
plt.show()

print(f"Mean weight: {sampled_weights.mean():.4f} (should be ~1.0 for unbiased estimation)")

---
## 6.4 — Noise Prediction Forward Pass

Let's walk through the forward pass step by step to build intuition for what happens inside `train_step`.

The key shape flow:

- **`x_0`**: `(B, 1, 28, 28)` — a batch of clean MNIST images
- **`t`**: `(B,)` — a random timestep per image
- **`x_t`**: `(B, 1, 28, 28)` — the noisy version (same shape as input)
- **`noise_pred`**: `(B, 1, 28, 28)` — the UNet's guess of what noise was added (same shape again)

Think of the UNet as a **noise detective** — given a corrupted image and the noise level, it tries to figure out exactly what noise was mixed in.

In [ ]:
torch.manual_seed(42)
demo_model = UNet(image_channels=1, base_channels=64, channel_mults=(1, 2, 4)).to(device)
schedule = cosine_schedule(1000)
sched_device = prepare_schedule(schedule, device)

# Step 1: Get a clean image batch
x_0, _ = next(iter(train_loader))  # (64, 1, 28, 28)
x_0 = x_0.to(device)
print(f"1. Clean images x_0:          {x_0.shape}")

# Step 2: Sample random timesteps
t = torch.randint(0, 1000, (x_0.shape[0],), device=device)  # (64,)
print(f"2. Timesteps t:               {t.shape}, range [{t.min()}, {t.max()}]")

# Step 3: Forward diffusion — add noise
x_t, noise = q_sample(x_0, t, sched_device)  # (64, 1, 28, 28) each
print(f"3. Noisy images x_t:          {x_t.shape}")
print(f"   Ground-truth noise eps:    {noise.shape}")

# Step 4: Model prediction
demo_model.eval()
with torch.no_grad():
    noise_pred = demo_model(x_t, t)  # (64, 1, 28, 28)
print(f"4. Predicted noise eps_theta: {noise_pred.shape}")

# Step 5: Compute loss
loss = F.mse_loss(noise_pred, noise)
print(f"5. MSE loss:                  {loss.item():.4f}")

# Visualize one example
idx = 0
fig, axes = plt.subplots(1, 4, figsize=(12, 3))
titles = [f'x_0 (clean)', f'x_t (t={t[idx].item()})', 'True noise', 'Predicted noise']
tensors = [x_0[idx, 0], x_t[idx, 0], noise[idx, 0], noise_pred[idx, 0]]
for ax, title, tensor in zip(axes, titles, tensors):
    ax.imshow(tensor.cpu().numpy(), cmap='gray')
    ax.set_title(title, fontsize=10)
    ax.axis('off')
plt.suptitle('Forward Pass Walkthrough', fontsize=12)
plt.tight_layout()
plt.show()

del demo_model

### Exercise 6.4: Shape Verification

Shape mismatches are one of the most common bugs in deep learning. Let's build a sanity-check function.

**Your task:** Write `verify_forward_pass()` that:

1. Creates a UNet with `base_channels=32` (smaller, for speed)
2. Generates a random batch of shape `(4, 1, 28, 28)` and random timesteps of shape `(4,)`
3. Asserts the **output shape** matches the input shape
4. Asserts the **output dtype** matches the input dtype
5. Asserts all output values are **finite**

In [ ]:
# YOUR CODE HERE — Exercise 6.4

def verify_forward_pass() -> None:
    """Verify UNet forward pass shapes, dtypes, and finiteness."""
    torch.manual_seed(42)
    # ===================== YOUR CODE HERE =====================
    # 1. Create a small UNet (base_channels=32) and move to device
    # 2. Create random input x of shape (4, 1, 28, 28) and timesteps t of shape (4,)
    # 3. Run forward pass (use torch.no_grad() since we don't need gradients)
    # 4. Assert: output shape == input shape
    # 5. Assert: output dtype == input dtype
    # 6. Assert: all values are finite
    pass
    # ====================== END YOUR CODE ======================


# Tests — run this cell to check your work
verify_forward_pass()
print("Forward pass verification ✓")

In [ ]:
# ✅ SOLUTION — try the exercise above before running this

def verify_forward_pass() -> None:
    """Verify UNet forward pass shapes, dtypes, and finiteness."""
    torch.manual_seed(42)
    test_model = UNet(image_channels=1, base_channels=32, channel_mults=(1, 2, 4)).to(device)
    test_model.eval()

    x = torch.randn(4, 1, 28, 28, device=device)         # (4, 1, 28, 28)
    t = torch.randint(0, 1000, (4,), device=device)       # (4,)

    with torch.no_grad():
        out = test_model(x, t)                             # (4, 1, 28, 28)

    assert out.shape == x.shape, f"Shape mismatch: got {out.shape} but expected {x.shape} — is the UNet output conv correct?"
    assert out.dtype == x.dtype, f"Dtype mismatch: got {out.dtype} but expected {x.dtype} — check for accidental type casting"
    assert torch.isfinite(out).all(), "Output contains NaN/Inf — check normalization layers and skip connections"

    print(f"Input shape:  {x.shape}, dtype: {x.dtype}")
    print(f"Output shape: {out.shape}, dtype: {out.dtype}")
    print(f"All finite:   {torch.isfinite(out).all().item()}")

    del test_model

verify_forward_pass()
print("Forward pass verification ✓")

---
## 6.5 — Loss Computation and Backpropagation

The loss in DDPM is beautifully simple — just MSE between the true noise and the predicted noise:

```
L_simple = E_{t, x_0, eps} [ || eps - eps_theta(x_t, t) ||^2 ]
```

In words: average over all timesteps, all training images, and all noise samples — **how far off is the network's noise prediction?** That's the entire training signal.

### Key details for stable training

- **`F.mse_loss`** — standard L2 loss, simple and effective for noise prediction
- **Gradient clipping** — prevents gradient explosions; `clip_grad_norm_(params, 1.0)` is standard
- **`optimizer.zero_grad()`** — must clear old gradients before each backward pass
- **Correct ordering** — `zero_grad → forward → loss → backward → clip → step`. Getting this wrong causes subtle bugs

In [ ]:
# Demonstrate the full backward pass with gradient monitoring

torch.manual_seed(42)
loss_model = UNet(image_channels=1, base_channels=64, channel_mults=(1, 2, 4)).to(device)
optimizer = torch.optim.Adam(loss_model.parameters(), lr=2e-4)
schedule = cosine_schedule(1000)
sched_device = prepare_schedule(schedule, device)

x_0, _ = next(iter(train_loader))
x_0 = x_0.to(device)  # (64, 1, 28, 28)

# Step-by-step backward pass
loss_model.train()

# 1. Zero gradients
optimizer.zero_grad()

# 2. Sample timesteps and noise
t = torch.randint(0, 1000, (x_0.shape[0],), device=device)
x_t, noise = q_sample(x_0, t, sched_device)

# 3. Forward pass
noise_pred = loss_model(x_t, t)  # (64, 1, 28, 28)

# 4. Compute loss
loss = F.mse_loss(noise_pred, noise)
print(f"Loss value: {loss.item():.4f}")

# 5. Backward
loss.backward()

# Check gradient norms BEFORE clipping
grad_norm_before = torch.nn.utils.clip_grad_norm_(loss_model.parameters(), max_norm=float('inf'))
print(f"Gradient norm (before clipping): {grad_norm_before:.4f}")

# Re-run backward (need to re-zero and re-compute since clip_grad_norm_ modifies in place)
optimizer.zero_grad()
noise_pred = loss_model(x_t, t)
loss = F.mse_loss(noise_pred, noise)
loss.backward()

# 6. Clip gradients
grad_norm_clipped = torch.nn.utils.clip_grad_norm_(loss_model.parameters(), max_norm=1.0)
print(f"Gradient norm (after clipping):  {min(grad_norm_clipped.item(), 1.0):.4f}")

# 7. Optimizer step
optimizer.step()

print("Backward pass completed successfully.")
del loss_model, optimizer

---
## 6.6 — Exponential Moving Average (EMA)

EMA is a simple trick that dramatically improves sample quality. It maintains a shadow copy of model weights that is a smoothed version of the training weights:

```
shadow_param = decay * shadow_param + (1 - decay) * current_param
```

**Instead of using the weights from the very last gradient step (which can be noisy), EMA blends in each new update slowly.** A typical decay of **0.9999** means only 0.01% of each new update gets blended in — so the shadow weights change very gradually.

Why this matters for diffusion models:

- **Training weights** are noisy because they reflect only the most recent gradient updates
- **EMA weights** are smoothed over many steps, producing higher-quality samples
- At inference time, we swap in the EMA weights for better generation

In [ ]:
class EMA:
    """Exponential Moving Average of model parameters.
    
    Usage:
        ema = EMA(model, decay=0.9999)
        # In training loop: ema.update(model)
        # For inference: ema.apply_shadow(model), then model(...), then ema.restore(model)
    """

    def __init__(self, model: nn.Module, decay: float = 0.9999) -> None:
        self.decay = decay
        self.shadow: Dict[str, torch.Tensor] = {}
        self.backup: Dict[str, torch.Tensor] = {}
        # Initialize shadow parameters as copies of current parameters
        for name, param in model.named_parameters():
            if param.requires_grad:
                self.shadow[name] = param.data.clone()

    def update(self, model: nn.Module) -> None:
        """Update shadow parameters with current model parameters."""
        for name, param in model.named_parameters():
            if param.requires_grad:
                self.shadow[name] = (
                    self.decay * self.shadow[name] + (1.0 - self.decay) * param.data
                )

    def apply_shadow(self, model: nn.Module) -> None:
        """Replace model parameters with shadow parameters (for inference)."""
        self.backup = {}
        for name, param in model.named_parameters():
            if param.requires_grad:
                self.backup[name] = param.data.clone()
                param.data = self.shadow[name].clone()

    def restore(self, model: nn.Module) -> None:
        """Restore original model parameters (after inference)."""
        for name, param in model.named_parameters():
            if param.requires_grad and name in self.backup:
                param.data = self.backup[name].clone()
        self.backup = {}


# Demonstrate EMA behavior
torch.manual_seed(42)
demo = nn.Linear(10, 10)
ema_demo = EMA(demo, decay=0.99)

# Simulate parameter updates
original_weight = demo.weight.data[0, 0].item()
for _ in range(100):
    # Simulate a gradient step (random perturbation)
    demo.weight.data += torch.randn_like(demo.weight.data) * 0.1
    ema_demo.update(demo)

print(f"Original weight[0,0]:  {original_weight:.4f}")
print(f"Current weight[0,0]:   {demo.weight.data[0, 0].item():.4f}  (noisy from updates)")
print(f"EMA shadow[0,0]:       {ema_demo.shadow['weight'][0, 0].item():.4f}  (smoothed)")

del demo, ema_demo

### Exercise 6.5: EMA Convergence Test

Let's see EMA in action with a simple 1D experiment.

**Your task:**

1. Start a parameter at **0.0**
2. Each step, set it to a noisy version of target value **5.0** (e.g., `5.0 + randn * 0.5`)
3. Track the raw parameter and EMA parameter over **200 steps** with `decay=0.99`
4. Plot both — the EMA line should be smoother and converge to ~5.0

- The raw values will jump around the target noisily
- The EMA should trace a smooth curve that steadily approaches 5.0

In [ ]:
# YOUR CODE HERE — Exercise 6.5

torch.manual_seed(42)
target = 5.0
decay = 0.99

raw_values = []
ema_values = []

# ===================== YOUR CODE HERE =====================
# Loop 200 steps:
#   1. Set param to noisy target: 5.0 + randn * 0.5
#   2. Update EMA shadow: shadow = decay * shadow + (1 - decay) * param
#   3. Append both to their lists
pass
# ====================== END YOUR CODE ======================


# Tests — run this cell to check your work
assert len(raw_values) == 200, f"Expected 200 steps but got {len(raw_values)} — check your loop range"
assert len(ema_values) == 200, f"Expected 200 EMA values but got {len(ema_values)} — are you appending the shadow value each step?"
assert abs(ema_values[-1] - target) < 0.5, f"EMA final value is {ema_values[-1]:.2f} — should be near {target}. Check the EMA formula: shadow = decay * shadow + (1 - decay) * param"

plt.figure(figsize=(10, 4))
plt.plot(raw_values, alpha=0.5, label='Raw parameter (noisy)', linewidth=0.8)
plt.plot(ema_values, label=f'EMA (decay={decay})', linewidth=2)
plt.axhline(y=target, color='r', linestyle='--', label=f'Target = {target}')
plt.xlabel('Step')
plt.ylabel('Value')
plt.title('EMA Smoothing Demonstration')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Final EMA value: {ema_values[-1]:.4f} (target: {target})")
print("EMA convergence ✓")

In [ ]:
# ✅ SOLUTION — try the exercise above before running this

torch.manual_seed(42)
target = 5.0
decay = 0.99
shadow = 0.0

raw_values = []
ema_values = []

for step in range(200):
    noisy_val = target + torch.randn(1).item() * 0.5
    shadow = decay * shadow + (1 - decay) * noisy_val
    raw_values.append(noisy_val)
    ema_values.append(shadow)

plt.figure(figsize=(10, 4))
plt.plot(raw_values, alpha=0.5, label='Raw parameter (noisy)', linewidth=0.8)
plt.plot(ema_values, label=f'EMA (decay={decay})', linewidth=2)
plt.axhline(y=target, color='r', linestyle='--', label=f'Target = {target}')
plt.xlabel('Step')
plt.ylabel('Value')
plt.title('EMA Smoothing Demonstration')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Final raw value: {raw_values[-1]:.4f}")
print(f"Final EMA value: {ema_values[-1]:.4f}")
print(f"EMA is closer to target: {abs(ema_values[-1] - target) < abs(raw_values[-1] - target)}")

---
## 6.7 — Learning Rate Schedules

A good LR schedule for diffusion training combines two phases:

### Warmup (first N steps)

Ramp the learning rate from 0 to the peak value. This prevents early instability when gradients are large and the model hasn't learned anything yet.

### Cosine decay (remaining steps)

Smoothly decrease the LR to near 0 following a cosine curve. This allows fine-grained convergence in the later stages of training.

```
lr(step) = {
  peak_lr * (step / warmup_steps)                            if step < warmup_steps
  peak_lr * 0.5 * (1 + cos(pi * (step - warmup) / decay))   otherwise
}
```

Intuitively: take big steps early to make progress, then smaller and smaller steps as you get closer to a good solution.

In [ ]:
def get_warmup_cosine_scheduler(
    optimizer: torch.optim.Optimizer,
    warmup_steps: int,
    total_steps: int,
) -> torch.optim.lr_scheduler.LambdaLR:
    """Warmup + cosine decay learning rate scheduler."""

    def lr_lambda(current_step: int) -> float:
        if current_step < warmup_steps:
            return current_step / max(1, warmup_steps)
        progress = (current_step - warmup_steps) / max(1, total_steps - warmup_steps)
        return 0.5 * (1.0 + math.cos(math.pi * progress))

    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)


# Visualize the schedule
dummy_model = nn.Linear(10, 10)
dummy_opt = torch.optim.Adam(dummy_model.parameters(), lr=2e-4)
total_steps = 5000
warmup_steps = 500
scheduler = get_warmup_cosine_scheduler(dummy_opt, warmup_steps, total_steps)

lrs = []
for step in range(total_steps):
    lrs.append(dummy_opt.param_groups[0]['lr'])
    scheduler.step()

plt.figure(figsize=(10, 4))
plt.plot(lrs)
plt.axvline(x=warmup_steps, color='r', linestyle='--', alpha=0.5, label=f'Warmup ends ({warmup_steps} steps)')
plt.xlabel('Training Step')
plt.ylabel('Learning Rate')
plt.title('Warmup + Cosine Decay Schedule')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Peak LR: {max(lrs):.6f}")
print(f"Final LR: {lrs[-1]:.6f}")

del dummy_model, dummy_opt

---
## 6.8 — Common Training Pitfalls

Before we start the full training run, let's go over the failure modes you're most likely to encounter. Bookmark this table — you'll come back to it.

| Symptom | Likely Cause | Fix |
|---|---|---|
| Loss spikes | Learning rate too high, or a bad batch | Reduce LR, add gradient clipping, increase warmup |
| Loss plateaus early | Model too small, LR too low, or data issue | Increase model capacity (`base_channels`), tune LR |
| NaN loss | Numerical overflow (often in attention or norm layers) | Check for division by zero in schedule, use `float32`, add `eps` to denominators |
| Mode collapse | Model only generates one or a few modes | Train longer, check data diversity, verify no label leakage |
| Blurry samples | Undertrained, or loss doesn't weight fine details | Train longer, try L1 loss or perceptual loss in addition to MSE |
| Slow convergence | Batch size too small, poor schedule choice | Increase batch size, switch to cosine schedule, use EMA |
| Memory errors | Model or batch too large for device | Reduce `base_channels`, reduce batch size, use gradient accumulation |

Let's see what happens when we deliberately pick a bad learning rate.

In [ ]:
# Demonstration: what happens with a bad learning rate

torch.manual_seed(42)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
learning_rates = [1e-2, 2e-4, 1e-6]
labels = ['LR=1e-2 (too high)', 'LR=2e-4 (good)', 'LR=1e-6 (too low)']

for ax, lr, label in zip(axes, learning_rates, labels):
    torch.manual_seed(42)
    m = UNet(image_channels=1, base_channels=64, channel_mults=(1, 2, 4)).to(device)
    o = torch.optim.Adam(m.parameters(), lr=lr)
    losses = []
    loader_iter = iter(train_loader)

    for step in range(30):
        try:
            batch, _ = next(loader_iter)
        except StopIteration:
            loader_iter = iter(train_loader)
            batch, _ = next(loader_iter)
        batch = batch.to(device)
        loss_val = train_step(m, batch, o, sched_device)
        losses.append(loss_val)

    ax.plot(losses)
    ax.set_title(label, fontsize=10)
    ax.set_xlabel('Step')
    ax.set_ylabel('Loss')
    ax.grid(True, alpha=0.3)
    del m, o

plt.suptitle('Effect of Learning Rate on Training', fontsize=12)
plt.tight_layout()
plt.show()

---
## 6.9 — Monitoring Training

Loss curves alone are **not sufficient** for evaluating diffusion models. You also need to inspect generated samples periodically.

- **Loss curve** — should decrease smoothly. A flat or rising curve means something is wrong
- **Sample grids** — generate a small grid of images every N steps. Early samples will be noise; as training progresses, structure should emerge (first rough shapes, then recognizable digits)

We'll use `ddpm_sample()` from `utils/diffusion.py` to periodically generate samples during training. We already covered the full sampling details in Module 5 — here we just use it as a monitoring tool.

In [ ]:
# Quick test: generate samples from an untrained model (should be pure noise)
torch.manual_seed(42)
untrained_model = UNet(image_channels=1, base_channels=64, channel_mults=(1, 2, 4)).to(device)
untrained_model.eval()

# Use ddpm_sample from utils — only 50 steps for a quick preview
noise_samples = ddpm_sample(untrained_model, sched_device, shape=(8, 1, 28, 28), device=device)
show_images(noise_samples, nrow=8, title='Samples from Untrained Model (should be noise)')

del untrained_model
print("Sampling functions ready for training monitoring.")

---
## Capstone — Full Training on MNIST

Now let's assemble everything into a complete training loop. This is where all the pieces come together — data loading, forward diffusion, noise prediction, EMA, and LR scheduling.

| Setting | Value |
|---|---|
| Dataset | MNIST 28×28, normalized to [-1, 1] |
| Schedule | Cosine, T=1000 |
| Optimizer | Adam, peak LR=2e-4 |
| Gradient clipping | max_norm=1.0 |
| EMA decay | 0.9999 |
| LR schedule | 500-step warmup + cosine decay |
| Training steps | 5000 |
| Sample generation | Every 1000 steps |
| Checkpoint | Saved at end to `checkpoints/ddpm_mnist.pt` |

This will take a few minutes on a GPU and longer on CPU/MPS. Watch the loss curve and sample grids — you should see structure emerge around step 2000–3000.

In [ ]:
# ---- Configuration ----
NUM_TIMESTEPS = 1000
TOTAL_STEPS = 5000
WARMUP_STEPS = 500
BATCH_SIZE = 64
LEARNING_RATE = 2e-4
EMA_DECAY = 0.9999
GRAD_CLIP = 1.0
SAMPLE_EVERY = 1000
NUM_SAMPLES = 16

# ---- Setup ----
torch.manual_seed(42)

model = UNet(
    image_channels=1,
    base_channels=64,
    channel_mults=(1, 2, 4),
    num_res_blocks=2,
    attention_resolutions=(7,),
    dropout=0.0,
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
scheduler = get_warmup_cosine_scheduler(optimizer, WARMUP_STEPS, TOTAL_STEPS)
schedule = cosine_schedule(NUM_TIMESTEPS)
sched_device = prepare_schedule(schedule, device)
ema = EMA(model, decay=EMA_DECAY)

total_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {total_params:,}")
print(f"Device: {device}")
print(f"Training for {TOTAL_STEPS} steps...")
print()

In [ ]:
# ---- Training Loop ----
loss_history = []
lr_history = []
loader_iter = iter(train_loader)

pbar = tqdm(range(TOTAL_STEPS), desc='Training')
for step in pbar:
    # Get batch (cycle through dataset)
    try:
        x_0, _ = next(loader_iter)
    except StopIteration:
        loader_iter = iter(train_loader)
        x_0, _ = next(loader_iter)

    x_0 = x_0.to(device)  # (B, 1, 28, 28)
    batch_size = x_0.shape[0]

    # --- DDPM Algorithm 1 ---
    model.train()
    optimizer.zero_grad()

    t = torch.randint(0, NUM_TIMESTEPS, (batch_size,), device=device)  # (B,)
    noise = torch.randn_like(x_0)  # (B, 1, 28, 28)

    sqrt_alpha_bar = sched_device['sqrt_alphas_cumprod'][t][:, None, None, None]  # (B, 1, 1, 1)
    sqrt_one_minus = sched_device['sqrt_one_minus_alphas_cumprod'][t][:, None, None, None]  # (B, 1, 1, 1)
    x_t = sqrt_alpha_bar * x_0 + sqrt_one_minus * noise  # (B, 1, 28, 28)

    noise_pred = model(x_t, t)  # (B, 1, 28, 28)
    loss = F.mse_loss(noise_pred, noise)

    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
    optimizer.step()
    scheduler.step()
    ema.update(model)

    loss_val = loss.item()
    loss_history.append(loss_val)
    lr_history.append(optimizer.param_groups[0]['lr'])

    # Update progress bar
    if step % 50 == 0:
        avg_loss = np.mean(loss_history[-50:]) if len(loss_history) >= 50 else np.mean(loss_history)
        pbar.set_postfix({'loss': f'{avg_loss:.4f}', 'lr': f'{lr_history[-1]:.2e}'})

    # Generate samples periodically
    if (step + 1) % SAMPLE_EVERY == 0:
        print(f"\nStep {step + 1}: generating samples with EMA weights...")
        ema.apply_shadow(model)
        samples = ddpm_sample(model, sched_device, shape=(NUM_SAMPLES, 1, 28, 28), device=device)
        show_images(samples, nrow=4, title=f'Samples at Step {step + 1}')
        ema.restore(model)

print(f"\nTraining complete. Final avg loss: {np.mean(loss_history[-100:]):.4f}")

In [ ]:
# ---- Plot Training Curves ----

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Loss curve (smoothed)
window = 100
if len(loss_history) >= window:
    smoothed = np.convolve(loss_history, np.ones(window) / window, mode='valid')
    axes[0].plot(smoothed, linewidth=0.8)
else:
    axes[0].plot(loss_history, linewidth=0.8)
axes[0].set_xlabel('Step')
axes[0].set_ylabel('MSE Loss (smoothed)')
axes[0].set_title('Training Loss')
axes[0].grid(True, alpha=0.3)

# LR curve
axes[1].plot(lr_history, linewidth=0.8, color='tab:orange')
axes[1].set_xlabel('Step')
axes[1].set_ylabel('Learning Rate')
axes[1].set_title('Learning Rate Schedule')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Final smoothed loss: {np.mean(loss_history[-100:]):.4f}")

In [ ]:
# ---- Final Samples with EMA ----

print("Generating final samples using EMA weights...")
ema.apply_shadow(model)
final_samples = ddpm_sample(model, sched_device, shape=(16, 1, 28, 28), device=device)
show_images(final_samples, nrow=4, title='Final Samples (EMA weights)')
ema.restore(model)
print("If you see recognizable digits, your training pipeline is working! 🎉")

In [ ]:
# ---- Save Checkpoint ----

os.makedirs('checkpoints', exist_ok=True)
checkpoint_path = 'checkpoints/ddpm_mnist.pt'

torch.save({
    'model_state_dict': model.state_dict(),
    'ema_shadow': ema.shadow,
    'optimizer_state_dict': optimizer.state_dict(),
    'config': {
        'image_channels': 1,
        'base_channels': 64,
        'channel_mults': (1, 2, 4),
        'num_res_blocks': 2,
        'attention_resolutions': (7,),
        'num_timesteps': NUM_TIMESTEPS,
    },
    'training_info': {
        'total_steps': TOTAL_STEPS,
        'final_loss': np.mean(loss_history[-100:]),
        'ema_decay': EMA_DECAY,
    },
    'loss_history': loss_history,
}, checkpoint_path)

file_size_mb = os.path.getsize(checkpoint_path) / (1024 * 1024)
print(f"Checkpoint saved to: {checkpoint_path}")
print(f"File size: {file_size_mb:.1f} MB")

# Verify checkpoint loads correctly
checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
print(f"Checkpoint keys: {list(checkpoint.keys())}")
print(f"Config: {checkpoint['config']}")
print(f"Final loss: {checkpoint['training_info']['final_loss']:.4f}")

---
## Summary

You've built a complete DDPM training pipeline from scratch — and you have a trained model to show for it! Here's what we covered:

| Section | Key Takeaway |
|---|---|
| 6.1 Data Loading | Normalize images to [-1, 1] to match Gaussian noise assumptions |
| 6.2 Algorithm 1 | The core loop: sample image, sample timestep, add noise, predict noise, MSE loss |
| 6.3 Timestep Sampling | Uniform sampling works well; importance sampling is a refinement |
| 6.4 Forward Pass | UNet takes `(x_t, t)` and outputs noise prediction with the same shape |
| 6.5 Loss and Backprop | MSE loss + gradient clipping + correct zero_grad/backward/step ordering |
| 6.6 EMA | Smoothed weights produce better samples; use decay=0.9999 |
| 6.7 LR Schedules | Warmup prevents early instability; cosine decay enables fine convergence |
| 6.8 Pitfalls | Know the common failure modes and their fixes |
| 6.9 Monitoring | Always inspect generated samples — loss alone is not sufficient |

The saved checkpoint at `checkpoints/ddpm_mnist.pt` is ready for Module 7 (sampling) and Module 8 (advanced techniques). Let's go generate some images!